# 11주차 과제
빅데이터프로그래밍 · 통계학과

**이름:**
**학번:**

## 과제 내용
사인파의 주기와 입력 구간 길이를 변경하고, RNN과 LSTM의 결과를 비교합니다.

## 제출 방법
모든 셀을 실행해 출력과 그래프가 보이는 상태로 저장한 뒤 `week11_학번_이름.ipynb` 로 제출합니다.


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import time

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

# 고정 조건 — 바꾸지 마세요
T, EPOCHS, BATCH, HIDDEN, LR, SEED = 1000, 30, 32, 64, 1e-3, 42
loss_fn = nn.MSELoss()


## 공통 함수
아래를 그대로 쓰고 주기·구간·모델만 바꿔 실험하세요.


In [ ]:
def make_series(freq=0.05, noise=0.05, T=T, seed=SEED):
    np.random.seed(seed)
    t = np.arange(T)
    return np.sin(t * freq) + np.random.randn(T) * noise


class SeriesDataset(Dataset):
    def __init__(self, arr, seq_len):
        self.arr = torch.tensor(arr, dtype=torch.float32)
        self.seq_len = seq_len
    def __len__(self):
        return len(self.arr) - self.seq_len
    def __getitem__(self, i):
        return (self.arr[i:i+self.seq_len].unsqueeze(-1),
                self.arr[i+self.seq_len].unsqueeze(-1))


def make_loaders(series, seq_len):
    n = int(len(series) * 0.8)
    return (DataLoader(SeriesDataset(series[:n], seq_len), batch_size=BATCH, shuffle=True),
            DataLoader(SeriesDataset(series[n:], seq_len), batch_size=BATCH, shuffle=False))


class Forecast(nn.Module):
    def __init__(self, kind="lstm", hidden=HIDDEN, num_layers=1):
        super().__init__()
        layer = {"rnn": nn.RNN, "lstm": nn.LSTM, "gru": nn.GRU}[kind]
        self.rnn = layer(1, hidden, num_layers=num_layers, batch_first=True)
        self.fc = nn.Linear(hidden, 1)
    def forward(self, x):
        out, _ = self.rnn(x)
        return self.fc(out[:, -1, :])


def run(kind, series, seq_len, epochs=EPOCHS, lr=LR, seed=SEED, **kw):
    torch.manual_seed(seed)
    tr, va = make_loaders(series, seq_len)
    model = Forecast(kind, **kw).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    start = time.time()
    for _ in range(epochs):
        model.train()
        for xb, yb in tr:
            xb, yb = xb.to(device), yb.to(device)
            loss = loss_fn(model(xb), yb)
            opt.zero_grad(); loss.backward(); opt.step()
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for xb, yb in va:
            preds.append(model(xb.to(device)).cpu()); trues.append(yb)
    p = torch.cat(preds).squeeze().numpy(); t = torch.cat(trues).squeeze().numpy()
    return {"pred": p, "true": t,
            "mae": float(np.abs(p-t).mean()),
            "rmse": float(np.sqrt(((p-t)**2).mean())),
            "time": time.time()-start,
            "params": sum(q.numel() for q in model.parameters())}


## 문제 1. LSTM 기본 실험 (20점)
`freq=0.05`, `seq_len=40` 으로 LSTM을 학습시키고 아래를 제출하세요.

- 학습·검증 손실 곡선
- 실제 값과 예측 값을 겹쳐 그린 선 그래프
- MAE와 RMSE


In [ ]:
# 답안


## 문제 2. 입력 구간 길이 변경 (25점)
`seq_len` 을 **5 · 10 · 20 · 40 · 80** 로 바꿔 각각 학습하고 RMSE를 비교하세요.

- 구간 길이별 RMSE를 표와 꺾은선 그래프로
- 표본 수도 함께 적으세요 (구간이 길면 표본이 줄어듭니다)


In [ ]:
# 답안


## 문제 3. 사인파 주기 변경 (25점)
`freq` 를 **0.02 · 0.05 · 0.1 · 0.3** 으로 바꿔 각각 학습하고 RMSE를 비교하세요.

각 주기가 몇 시점인지(`2π/freq`), 입력 구간에 주기가 몇 개 담기는지도 함께 적으세요.


In [ ]:
# 답안


## 문제 4. RNN과 LSTM 비교 (20점)
같은 조건에서 **RNN과 LSTM**을 비교하세요. `seq_len` 을 **10 · 40 · 100** 으로 바꿔 각각 비교합니다.

| seq_len | RNN RMSE | LSTM RMSE | 파라미터 (RNN / LSTM) | 시간 |

예측 그래프도 나란히 그리세요.


In [ ]:
# 답안


## 문제 5. 해석 (10점)
아래 세 가지를 각각 두세 줄로 적으세요.

1. 입력 구간을 길게 하면 항상 좋아지는가 — 이 실험에서는 어땠나
2. 사인파 주기가 짧을 때(빠르게 흔들릴 때) 예측이 어려워지는 이유
3. 구간이 길어질수록 RNN과 LSTM의 차이가 어떻게 변했는가


**답:**

1.

2.

3.


## 문제 6. 연속 예측 (보너스)
예측값을 다시 입력으로 넣어 100 시점을 이어서 예측하고, 실제 값과 겹쳐 그리세요. 오차가 어떻게 누적되는지 설명하세요.


In [ ]:
# 답안
